# Multi-Agent Financial Analysis System
**AAI-520 Final Project | University of San Diego**
Andre Hoth & Ken Lai

---

## Project Overview

This notebook demonstrates an autonomous Investment Research Agent that, given a stock ticker symbol, plans its own research strategy, retrieves financial and news data from multiple sources, analyzes it with specialized agents, synthesizes a coherent investment analysis, evaluates and refines its own output, and retains memory across runs to improve future analyses. Content routing, which directs content to the right specialist, is implemented and demonstrated separately in Section 6; the full pipeline invokes the three specialists directly.

### Rubric Requirements Demonstrated

| Requirement | Component | Module |
|---|---|---|
| Agent Function: Planning | InvestmentResearchAgent | agents/investment_agent.py |
| Agent Function: Dynamic Tool Use | Tools Layer | tools/ |
| Agent Function: Self-Reflection | InvestmentResearchAgent | agents/investment_agent.py |
| Agent Function: Memory | AgentMemory | memory/agent_memory.py |
| Workflow 1: Prompt Chaining | NewsProcessingChain | workflows/prompt_chain.py |
| Workflow 2: Routing | ContentRouter | workflows/router.py |
| Workflow 3: Evaluator-Optimizer | EvaluatorOptimizer | workflows/evaluator_optimizer.py |

### GitHub Repository
[aai-520-multi-agent-financial](https://github.com/andrehoth/aai-520-multi-agent-financial)

---

## 1. Setup

In [ ]:
# ── Imports and Configuration ─────────────────────────────────────────────────
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Tools
from tools.price_data import fetch_price_history, fetch_price_summary
from tools.news_data import fetch_news
from tools.fundamentals import fetch_earnings, fetch_income_statement
from tools.macro_data import fetch_macro_indicators
from tools.llm import call_llm, LLM_PROVIDER, LLM_MODEL

# Agents
from agents.schema import empty_analysis, ANALYSIS_SCHEMA, ContentType
from agents.earnings_agent import EarningsAnalyzer
from agents.news_agent import NewsAnalyzer
from agents.market_agent import MarketAnalyzer
from agents.investment_agent import InvestmentResearchAgent

# Workflows
from workflows.prompt_chain import NewsProcessingChain
from workflows.router import ContentRouter

# Memory
from memory.agent_memory import save_memory, load_memory, clear_memory

# AAPL selected for broad coverage across all four data sources (NewsAPI,
# Alpha Vantage, yfinance, FRED) to minimize empty fields on free-tier APIs.
TICKER = "AAPL"

print(f"Ticker:       {TICKER}")
print(f"LLM Provider: {LLM_PROVIDER}")
print(f"LLM Model:    {LLM_MODEL}")
print(f"Timestamp:    {datetime.now().isoformat()}")

---

## 2. Design Rationale: Assignment Requirements to Architecture

The following mapping drives all architectural decisions in this notebook.
All data sources are drawn from the assignment brief's recommended list.

### Rubric Requirements to Specialist Agents to Data Sources

The routing workflow requires three specialist agents. The agent responsibilities
determine the data domains, which in turn determine the tool files.

| Rubric Requirement | Specialist Agent | Data Domain | Tool | Source |
|---|---|---|---|---|
| Routing: Earnings | EarningsAnalyzer | Fundamentals | fundamentals.py | Alpha Vantage |
| Routing: News | NewsAnalyzer | Financial news | news_data.py | NewsAPI |
| Routing: Market | MarketAnalyzer | Price data | price_data.py | yfinance |
| Routing: Market | MarketAnalyzer | Macro context | macro_data.py | FRED API |

### Agent Function: Dynamic Tool Use

The assignment requires the agent to "use tools dynamically (APIs, datasets,
retrieval)." Each specialist agent calls the tools for its own data domain at
run time, retrieving live data for the requested ticker from four external
APIs. Tool selection follows agent responsibility boundaries, so each
specialist uses only the tools relevant to its analytical domain. The
instructor confirmed that an explicit RAG implementation is not required;
the four API tool calls satisfy the retrieval requirement in scope
(Mokhtari Jadid, K., personal communication, September 2026).

### Inter-Agent Schema Decision

A narrative dict is used to pass analysis between components rather than a typed
structured schema. Each specialist agent produces a narrative string. Synthesis
weaves those strings into a final analysis. The evaluator scores the synthesis
narrative using an LLM rubric prompt. See docs/architecture.md Section 6.1 for
the full design decision and tradeoffs.

---

## 3. Inter-Agent Data Schema

The shared analysis dict is defined in `agents/schema.py`. All pipeline
components read from and write to this dict. Fields are initialized to empty
values upfront so that any stage can safely reference any key without checking
existence first, and a falsy check reliably indicates whether a stage has run.

In [ ]:
# ── Inter-Agent Data Schema ───────────────────────────────────────────────────
# Demonstrates the shared analysis dict that flows through the entire pipeline.
# Fields are populated progressively as each pipeline stage completes.
# See agents/schema.py for full field definitions and design decision rationale.

analysis = empty_analysis(TICKER)

print(f"Analysis dict initialized for {TICKER}:")
for key, value in analysis.items():
    print(f"  {key}: {repr(value)}")

print(f"\nContentType values available to router: {[ct.value for ct in ContentType]}")

---

## 4. Agent Function: Dynamic Tool Use

The four data tools are called at run time by the specialist agents and the news processing chain, each retrieving live data for the requested ticker. The InvestmentResearchAgent orchestrates these tools through its specialist agents and supporting workflows. Although the research planner generates an analysis strategy, the current implementation follows a predefined execution sequence rather than dynamically selecting tools based on the plan. Each tool wraps a specific API, and all data in this notebook is retrieved live at run time rather than mocked.

### Tool 1: Price Data (yfinance)

Fetches OHLCV price history and summary statistics for the demonstration ticker.
Serves the MarketAnalyzer specialist agent.

In [ ]:
# ── Tool 1: Price Data ────────────────────────────────────────────────────────
# yfinance wrapper; no API key required (open source library, no registration)
# Downstream consumer: MarketAnalyzer (agents/market_agent.py)

price_summary = fetch_price_summary(TICKER)
price_history = fetch_price_history(TICKER, period="3mo")

print(f"Price Summary for {TICKER}:")
for key, value in price_summary.items():
    print(f"  {key}: {value}")

print(f"\nPrice History (last 5 trading days):")
print(price_history.tail())

In [ ]:
# ── Visualization: Price History ──────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.plot(price_history.index, price_history["Close"], color="steelblue", linewidth=1.5)
ax1.axhline(
    y=price_summary["week_52_high"],
    color="green",
    linestyle="--",
    alpha=0.7,
    label="52-week high",
)
ax1.axhline(
    y=price_summary["week_52_low"],
    color="red",
    linestyle="--",
    alpha=0.7,
    label="52-week low",
)
ax1.set_ylabel("Price (USD)")
ax1.set_title(f"{TICKER} Price History (3 months)")
ax1.legend()

ax2.bar(
    price_history.index, price_history["Volume"] / 1e6, color="steelblue", alpha=0.6
)
ax2.axhline(
    y=price_summary["avg_volume"] / 1e6,
    color="orange",
    linestyle="--",
    alpha=0.7,
    label="Avg volume",
)
ax2.set_ylabel("Volume (millions)")
ax2.set_xlabel("Date")
ax2.legend()

plt.tight_layout()
plt.show()

Price history and volume data fed to MarketAnalyzer for trend and momentum analysis.

### Tool 2: Financial News (NewsAPI)

Fetches recent English-language financial news articles for the demonstration
ticker. Serves the NewsProcessingChain and NewsAnalyzer specialist agent.
Language filtering via langdetect and query refinement are applied at fetch time.

In [ ]:
# ── Tool 2: Financial News ────────────────────────────────────────────────────
# NewsAPI wrapper; API key required (free tier, register at newsapi.org)
# Downstream consumer: NewsProcessingChain -> NewsAnalyzer (agents/news_agent.py)
# Language filtering and query refinement applied at fetch time

articles = fetch_news(TICKER)

print(f"News articles fetched for {TICKER}: {len(articles)}")
print()
for i, article in enumerate(articles[:3], 1):
    print(f"Article {i}:")
    print(f"  Title:  {article['title']}")
    print(f"  Source: {article['source']}")
    print(f"  Date:   {article['published_at']}")
    print()

### Tool 3: Earnings and Fundamentals (Alpha Vantage)

Fetches quarterly earnings history and income statement data for the
demonstration ticker. Serves the EarningsAnalyzer specialist agent.
Results are limited to the 8 most recent quarters for relevance.
Alpha Vantage's free tier allows 25 requests per day, so successful responses
are cached for the session. When the requests succeed, a full notebook run
requests earnings and income statement data once each.


In [ ]:
# ── Tool 3: Earnings and Fundamentals ────────────────────────────────────────
# Alpha Vantage wrapper; API key required (free tier, register at alphavantage.co)
# Downstream consumer: EarningsAnalyzer (agents/earnings_agent.py)
# Limited to 8 most recent quarters. Successful responses are cached for the
# session, so the specialist agents later in this notebook reuse this data
# instead of spending more of the free tier's 25 requests per day.
earnings = fetch_earnings(TICKER)
income = fetch_income_statement(TICKER)

if earnings:
    print(f"Earnings history for {TICKER} (most recent 3 quarters):")
    for q in earnings[:3]:
        print(
            f"  {q['fiscalDateEnding']}: EPS {q['reportedEPS']} vs estimate "
            f"{q['estimatedEPS']} (surprise: {q['surprisePercentage']}%)"
        )
else:
    # The tool prints the reason above (for example, a daily request limit)
    print("Earnings data unavailable; see the warning above for the cause.")

### Tool 4: Macroeconomic Indicators (FRED API)

Fetches macroeconomic indicators including the federal funds rate, CPI, and
unemployment rate. Serves the MarketAnalyzer specialist agent, providing macro
context that differentiates it from the EarningsAnalyzer.

In [ ]:
# ── Tool 4: Macroeconomic Indicators ─────────────────────────────────────────
# FRED API wrapper; API key required (free, register at fred.stlouisfed.org)
# Downstream consumer: MarketAnalyzer (agents/market_agent.py)
# Provides macro context distinct from company-level earnings data

macro = fetch_macro_indicators()

print("Macroeconomic indicators (most recent observation):")
for indicator, observations in macro.items():
    latest = observations[-1]
    print(f"  {indicator}: {latest['value']} ({latest['date']})")

In [ ]:
# ── Visualization: Macroeconomic Indicators ───────────────────────────────────
# Shows fed funds rate, CPI, and unemployment rate over the analysis period.
# Three indicators are plotted separately due to different scales and units.
# Feeds MarketAnalyzer for macroeconomic context analysis.

fed_dates = [
    datetime.strptime(obs["date"], "%Y-%m-%d") for obs in macro["fed_funds_rate"]
]
fed_values = [obs["value"] for obs in macro["fed_funds_rate"]]

cpi_dates = [datetime.strptime(obs["date"], "%Y-%m-%d") for obs in macro["cpi"]]
cpi_values = [obs["value"] for obs in macro["cpi"]]

unemp_dates = [
    datetime.strptime(obs["date"], "%Y-%m-%d") for obs in macro["unemployment_rate"]
]
unemp_values = [obs["value"] for obs in macro["unemployment_rate"]]

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

ax1.plot(fed_dates, fed_values, color="steelblue", linewidth=1.5)
ax1.set_ylabel("Fed Funds Rate (%)")
ax1.set_title("Macroeconomic Indicators (2024 to present)")

ax2.plot(cpi_dates, cpi_values, color="darkorange", linewidth=1.5)
ax2.set_ylabel("CPI")

ax3.plot(unemp_dates, unemp_values, color="green", linewidth=1.5)
ax3.set_ylabel("Unemployment Rate (%)")
ax3.set_xlabel("Date")
ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

---

## 5. Workflow 1: Prompt Chaining

The NewsProcessingChain processes raw news articles through five sequential stages,
with each stage's output passed as input to the next. Ingest retrieves articles
from NewsAPI and Preprocess formats them as text; Classify, Extract, and
Summarize are separate LLM calls.

1. **Ingest** -- retrieve raw articles from NewsAPI via `tools/news_data.fetch_news()`
2. **Preprocess** -- convert articles into clean structured text
3. **Classify** -- filter for relevance to the target ticker
4. **Extract** -- identify investment-relevant facts and developments
5. **Summarize** -- produce a concise factual narrative

In [ ]:
# ── Workflow 1: Prompt Chaining ───────────────────────────────────────────────
# NewsProcessingChain runs five sequential stages (three of them LLM calls),
# each building on the previous stage's output. The chain is self-contained;
# news retrieval occurs inside ingest() via tools/news_data.fetch_news().
# See workflows/prompt_chain.py for full implementation.

chain = NewsProcessingChain(TICKER)

print("Stage 1: Ingest")
articles = chain.ingest()
print(f"  Retrieved {len(articles)} articles")
print()

print("Stage 2: Preprocess")
preprocessed = chain.preprocess(articles)
print(f"  Preprocessed text length: {len(preprocessed)} characters")
print()

print("Stage 3: Classify")
classified = chain.classify(preprocessed)
print(f"  Classified text length: {len(classified)} characters")
print()

print("Stage 4: Extract")
extracted = chain.extract(classified)
print(f"  Extracted text length: {len(extracted)} characters")
print()

print("Stage 5: Summarize")
summarized = chain.summarize(extracted)
print(f"  Summary length: {len(summarized)} characters")
print()
print("Chain output:")
print(summarized)

---

## 6. Workflow 2: Routing and Specialist Agents

The ContentRouter classifies incoming financial content by type and dispatches
to the appropriate specialist agent. Each specialist fetches its own data and
produces a narrative analysis. Routing is based on content type, not data source.
In the full pipeline (Section 11), the orchestrator invokes the three
specialists directly because each one fetches its own data; the router is
demonstrated here.


### Specialist Agent 1: EarningsAnalyzer

Analyzes quarterly earnings and income statement data. Dispatched by the router
when content is classified as EARNINGS.

In [ ]:
# ── EarningsAnalyzer ──────────────────────────────────────────────────────────
# Fetches earnings and income data via Alpha Vantage and produces narrative
# analysis covering trends, revenue growth, margins, and earnings surprises.
# Populates analysis["earnings_analysis"] in the shared analysis dict.

earnings_agent = EarningsAnalyzer(TICKER)
analysis = earnings_agent.run(analysis)

if analysis["earnings_analysis"]:
    print(f"Earnings analysis length: {len(analysis['earnings_analysis'])} characters")
    print()
    print(analysis["earnings_analysis"])
else:
    print(
        "Earnings analysis returned empty. Possible causes: API rate limit, "
        "network error, or invalid ticker. Check Alpha Vantage quota at "
        "alphavantage.co/support."
    )

In [ ]:
# ── Visualization: Earnings Surprise ─────────────────────────────────────────
# Shows how much reported earnings per share (EPS) differed from analyst estimates
# each quarter, as a percentage. Positive values mean reported EPS exceeded estimates.
# Sourced from Alpha Vantage via tools/fundamentals.py.
# Feeds EarningsAnalyzer for trend analysis.

dates = [q["fiscalDateEnding"] for q in reversed(earnings)]
surprises = [float(q["surprisePercentage"]) for q in reversed(earnings)]
colors = ["green" if s > 0 else "red" for s in surprises]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(dates, surprises, color=colors, alpha=0.8)
ax.axhline(y=0, color="black", linewidth=0.8)
ax.set_title(f"{TICKER} Earnings Surprise % by Quarter (most recent 8)")
ax.set_ylabel("Surprise %")
ax.set_xlabel("Fiscal Quarter End")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Earnings surprise data fed to EarningsAnalyzer for earnings trend analysis.

### Specialist Agent 2: MarketAnalyzer

Analyzes price trends, momentum, and macroeconomic context. Dispatched by the
router when content is classified as MARKET or MACRO.

In [ ]:
# ── MarketAnalyzer ────────────────────────────────────────────────────────────
# Fetches price history via yfinance and macro indicators via FRED API.
# Produces narrative analysis covering price trends, momentum, and how the
# macro environment relates to the stock.
# Populates analysis["market_analysis"] in the shared analysis dict.

market_agent = MarketAnalyzer(TICKER)
analysis = market_agent.run(analysis)

print(f"Market analysis length: {len(analysis['market_analysis'])} characters")
print()
print(analysis["market_analysis"])

### Specialist Agent 3: NewsAnalyzer

Processes recent financial news through the five-stage prompt chain and produces
a narrative analysis covering material developments, sentiment, implications,
and key uncertainties. Dispatched by the router when content is classified as NEWS.

In [ ]:
# ── NewsAnalyzer ──────────────────────────────────────────────────────────────
# Fetches news via NewsAPI, processes through all five chain stages, then
# produces narrative analysis covering material developments, sentiment,
# and implications for the stock.
# Populates analysis["news_analysis"] in the shared analysis dict.
# See workflows/prompt_chain.py for the full five-stage chain implementation.

chain_output = chain.run()
news_agent = NewsAnalyzer(TICKER)
analysis = news_agent.run(analysis, processed_news=chain_output)

print(f"News analysis length: {len(analysis['news_analysis'])} characters")
print()
print(analysis["news_analysis"])

### Routing Demonstration

The ContentRouter uses an LLM call at temperature 0.2 to classify content
and dispatch to the correct specialist. The following demonstrates classification
across all four content types: EARNINGS, NEWS, MARKET, and MACRO. MARKET and
MACRO content both route to the MarketAnalyzer.


In [ ]:
# ── Workflow 2: Routing Demonstration ────────────────────────────────────────
# ContentRouter classifies content into NEWS, EARNINGS, MARKET, or MACRO
# and dispatches to the matching specialist agent.
# Temperature 0.2 used for classification; deterministic output preferred.
# See workflows/router.py and agents/schema.py (ContentType enum).

router = ContentRouter(TICKER)

test_inputs = [
    "Apple reported EPS of 2.02, beating estimates of 1.88",
    "Apple faces a 2.7 billion lawsuit over App Store tracking rules",
    "Federal Reserve holds interest rates at 3.63 percent amid inflation concerns",
    "AAPL closed up 3 percent on trading volume of 60 million shares, above its 50-day average",
]

print("Router classification test:")
print()
for content in test_inputs:
    ct = router.classify(content)
    specialist = router.specialists[ct].__class__.__name__
    print(f"  Input:      {content[:60]}...")
    print(f"  Classified: {ct.value} -> {specialist}")
    print()

### Routing Dispatch

`route()` combines classification with dispatch: it classifies the content, then
runs the matching specialist on the shared analysis dict. A fresh dict is used
below so that the populated field shows which specialist received the content.
`tests/test_router.py` (Section 12) additionally verifies, with mocked
dependencies, that each content type reaches the correct specialist and
populates only that specialist's field.


In [ ]:
# ── Workflow 2: Routing Dispatch ─────────────────────────────────────────────
# route() classifies the content, then runs the matching specialist on the
# shared analysis dict. A fresh dict is passed in, so the field that gets
# populated shows which specialist received the content. This call fetches
# earnings data and makes LLM calls, so it takes a moment to run.
from agents.schema import empty_analysis

routed = router.route(
    "Apple reported EPS of 2.02, beating estimates of 1.88",
    empty_analysis(TICKER),
)

print()
print("Analysis fields after routing:")
for field in ("earnings_analysis", "news_analysis", "market_analysis"):
    # A non-empty string means the specialist that owns this field was run.
    status = f"{len(routed[field])} characters" if routed[field] else "empty"
    print(f"  {field}: {status}")

---

Sections 7 through 10 demonstrate each workflow pattern and agent function in
isolation. The system's memory module persists analysis results across runs so
the planner can address gaps identified in prior reflections. A pipeline run
(Run 1) is executed below to populate memory with an initial analysis, giving
the subsequent demos real prior context to work with. Section 11 then runs the
full pipeline two more times (Run 2 and Run 3) to demonstrate how memory
compounds across runs.

In [ ]:
# Run 1 (seeding run): populate memory so Sections 7 through 10 have prior
# context to demonstrate against. Section 11 continues the sequence as
# Run 2 and Run 3.
from agents.investment_agent import InvestmentResearchAgent
from memory.agent_memory import clear_memory

# Clear any memory left over from earlier sessions so Run 1 starts from a
# known state and the three-run progression is reproducible.
clear_memory(TICKER)

# Execute the full pipeline once and keep the result so Run 1's score can be
# compared with Runs 2 and 3 in Section 11.
agent = InvestmentResearchAgent(TICKER)
analysis_run1 = agent.run()
print(f"Run 1 complete. Memory seeded for {TICKER}.")

## 7. Workflow 3: Evaluator-Optimizer

The EvaluatorOptimizer evaluates the synthesized analysis on a 0-10 scale, then
refines it using the evaluator's feedback. In the pipeline, `run()` repeats this
cycle until the score reaches the quality threshold (7.0) or the maximum number
of iterations (3) is exhausted. A rubric-anchored prompt defines what each score
level means, and a 10-point interval scale was selected because interval scales
were more self-consistent than Likert or binary scales for most of the models
compared in a study of LLM evaluators (Lee et al., 2024). Because the evaluator
is itself an LLM, re-scoring identical text can return a different value.

The demonstration below uses the synthesis saved by Run 1. It first walks through
one explicit evaluate, refine, and re-evaluate cycle, then shows the loop control
that `run()` applies in the pipeline. The Step 1 score can differ from the score
the pipeline assigned to the same synthesis in Run 1 above.


In [ ]:
# Section 7 -- Workflow 3: Evaluator-Optimizer
# Demonstrate the generate -> evaluate -> refine -> re-evaluate cycle in
# isolation. The synthesis saved to memory by Run 1 stands in for the
# "generate" step, so the full pipeline does not need to run again.

from workflows.evaluator_optimizer import EvaluatorOptimizer
from memory.agent_memory import load_memory

# Load the analysis saved by the most recent pipeline run (Run 1 at this point)
prior = load_memory(TICKER)

if not prior:
    print(f"No prior analysis found for {TICKER} -- run the Run 1 cell above first.")
else:
    evaluator = EvaluatorOptimizer(TICKER)
    original = prior["synthesis"]

    # Step 1: evaluate the saved synthesis. evaluate() returns a 0-10 score and
    # the evaluator's written feedback.
    score_before, feedback = evaluator.evaluate(original)
    print("=== Step 1: Evaluate ===")
    print(f"Score: {score_before}/10")
    print(f"Feedback length: {len(feedback)} characters")
    print("Feedback (first 500 chars):")
    print(feedback[:500])

    # Step 2: refine the synthesis using that feedback. refine() makes a second
    # LLM call that revises the text to address the feedback.
    refined = evaluator.refine(original, feedback)
    print("\n=== Step 2: Refine ===")
    print(f"Synthesis length: {len(original)} -> {len(refined)} characters")

    # Step 3: re-evaluate the refined synthesis. The score is not guaranteed to
    # rise, because the evaluator is itself an LLM.
    score_after, _ = evaluator.evaluate(refined)
    print("\n=== Step 3: Re-evaluate ===")
    print(f"Score: {score_after}/10")

    # Loop control: run() applies the same cycle with a stopping rule. It
    # evaluates, stops once the score reaches the threshold, and otherwise
    # refines and re-evaluates up to max_iterations. A copy of the analysis is
    # passed so the saved memory entry is not modified.
    loop = EvaluatorOptimizer(TICKER, threshold=7.0, max_iterations=3)
    result = loop.run(dict(prior))
    print("\n=== Loop control (threshold 7.0, max 3 iterations) ===")
    print(f"Final score: {result['evaluation_score']}/10")

---

## 8. Agent Function: Planning

The InvestmentResearchAgent generates a multi-step research plan for the given
ticker before the specialist agents retrieve their respective data. Prior memory
is read at this stage so the planner can address gaps identified in previous
analyses.

In [ ]:
# Section 8 -- Agent Function: Planning
# Demonstrate the planning stage in isolation, showing how prior memory
# informs the research agenda for the current run.

from agents.investment_agent import InvestmentResearchAgent
from memory.agent_memory import load_memory

agent = InvestmentResearchAgent(TICKER)

# Show prior memory context the planner will reference
prior = load_memory(TICKER)
if prior:
    print(f"Prior memory found for {TICKER} (score: {prior['evaluation_score']}/10)")
    print(f"Planner will address gaps from: {prior['timestamp']}\n")
else:
    print(f"No prior memory for {TICKER} -- planner will generate baseline plan\n")

# Generate the research plan
plan = agent.plan(prior_memory=prior)
print("=== Research Plan ===")
print(plan)

---

## 9. Agent Function: Self-Reflection

After synthesis and evaluation, the InvestmentResearchAgent reflects on the quality of its own output, identifying what was covered well, what was missing, and priorities for the next run. The reflection narrative is persisted to memory for use in future runs.

In [ ]:
# Section 9 -- Agent Function: Self-Reflection
# Demonstrate the reflection stage in isolation using the saved analysis
# from the most recent pipeline run.

from agents.investment_agent import InvestmentResearchAgent
from memory.agent_memory import load_memory

agent = InvestmentResearchAgent(TICKER)

# Load the most recent analysis from memory to reflect on
prior = load_memory(TICKER)
if not prior:
    print(f"No prior analysis found for {TICKER} -- run the Run 1 cell above first.")
else:
    reflection = agent.reflect(prior)
    print("=== Self-Reflection ===")
    print(reflection)

---

## 10. Agent Function: Memory

The AgentMemory module persists completed analysis dicts to
`data/agent_memory.json`, keyed by ticker and capped at 5 entries per ticker.
On subsequent runs, the planner reads the most recent entry to inform its
research strategy and address prior gaps.

In [ ]:
# Section 10 -- Agent Function: Memory
# Demonstrate the memory module in isolation: inspect the persisted entry
# for the current ticker and show what the planner will use on the next run.

from memory.agent_memory import load_memory

# Load most recent memory entry for this ticker
entry = load_memory(TICKER)

if entry:
    print(f"Most recent memory entry for {TICKER}:")
    print(f"  Timestamp:        {entry['timestamp']}")
    print(f"  Evaluation score: {entry['evaluation_score']}/10")
    print(f"  Synthesis length: {len(entry['synthesis'])} characters")
    print(f"  Reflection saved: {len(entry['reflection'])} characters")
    print(f"\nThis entry will be passed to the planner on the next run")
    print(f"to address gaps identified in the reflection.")
else:
    print(f"No memory found for {TICKER} -- run the Run 1 cell above first.")

---

## 11. Full Pipeline: InvestmentResearchAgent

The InvestmentResearchAgent orchestrates the complete research pipeline in a
single `agent.run()` call. To demonstrate Agent Function: Memory, the pipeline
is run three times in total. The seeding run before Section 7 is Run 1.
Run 2 reads Run 1's memory and generates a research plan targeting prior gaps.
Run 3 reads Run 2's memory and reflection, allowing the planner to address
identified gaps across three compounding iterations.

Note: This notebook uses live API data. Re-execution will produce different
outputs as market data, news, and macro indicators change over time. The
memory file (`data/agent_memory.json`) is gitignored and not included in
the repository; it is created at runtime. The Run 1 cell clears any existing
memory for the ticker at the start of each full notebook run, so the
three-run progression always begins from a known state.

```
Ticker Input
    |
Planning (reads memory)
    |
Routing (Workflow 2): fetch articles -> classify_batch() -> NEWS / EARNINGS / MARKET buckets
    |
Prompt Chain (Workflow 1): Preprocess -> Classify -> Extract -> Summarize
(NEWS bucket only; output feeds NewsAnalyzer)
    |
Specialist Agents with Dynamic Tool Use (all three always run)
EarningsAnalyzer: structured Alpha Vantage data + EARNINGS bucket (extra_context)
MarketAnalyzer:   structured yfinance/FRED data  + MARKET bucket  (extra_context)
NewsAnalyzer:     prompt chain output (NEWS bucket processed through Workflow 1)
    |
Synthesis
    |
Evaluator-Optimizer (Workflow 3): evaluate -> refine -> re-evaluate
until score reaches threshold (default 7.0) or 3 iterations
    |
Self-Reflection
    |
Memory Write
    |
Final Investment Analysis Report
```

ContentRouter.classify_batch() (Workflow 2) runs inside the pipeline at Stage 2. All fetched articles are classified in a single LLM call and split into three buckets. NEWS-classified articles pass through the five-stage prompt chain (Workflow 1) and feed NewsAnalyzer. EARNINGS-classified articles are forwarded as extra_context to EarningsAnalyzer alongside structured Alpha Vantage data. MARKET and MACRO-classified articles are forwarded as extra_context to MarketAnalyzer alongside structured yfinance and FRED data. All three specialists always run.


In [ ]:
# ── Workflow 2 in the Pipeline: classify_batch() ─────────────────────────────
# Shows the routing stage that runs inside agent.run() at Stage 2.
# All articles are classified in a single LLM call and split into buckets
# before the prompt chain runs, so each specialist receives only the content
# relevant to its domain.

from tools.news_data import fetch_news
from workflows.router import ContentRouter
from agents.schema import ContentType

demo_router = ContentRouter(TICKER)
demo_articles = fetch_news(TICKER)

print(f"Articles fetched: {len(demo_articles)}\n")

if demo_articles:
    demo_labels = demo_router.classify_batch(demo_articles)

    print("Article classifications:")
    print()
    for article, label in zip(demo_articles, demo_labels):
        title = (article.get("title") or "")[:70]
        print(f"  [{label.value:<8}]  {title}")

    news_bucket = [
        a for a, l in zip(demo_articles, demo_labels) if l == ContentType.NEWS
    ]
    earnings_bucket = [
        a for a, l in zip(demo_articles, demo_labels) if l == ContentType.EARNINGS
    ]
    market_bucket = [
        a
        for a, l in zip(demo_articles, demo_labels)
        if l in (ContentType.MARKET, ContentType.MACRO)
    ]

    print()
    print("Bucket split:")
    print(f"  NEWS     -> prompt chain only          : {len(news_bucket)} articles")
    print(f"  EARNINGS -> EarningsAnalyzer (context) : {len(earnings_bucket)} articles")
    print(f"  MARKET   -> MarketAnalyzer   (context) : {len(market_bucket)} articles")
else:
    print("No articles retrieved -- NewsAPI quota may be exhausted.")

The EARNINGS bucket is empty in this run because no fetched articles report quarterly results, EPS figures, or earnings guidance. Apple's next earnings release is November 2, 2026. In runs that follow an earnings release, NewsAPI articles covering results and analyst reactions would classify as EARNINGS and appear in EarningsAnalyzer's prompt as extra_context alongside the structured Alpha Vantage data, which is confirmed present in Section 4.

### Run 2: Memory-Informed Analysis

Run 2 reads the memory saved by the seeding run (Run 1) and generates a research
plan that addresses gaps identified in the prior reflection. Results are
saved to memory for Run 3 to build on.


In [ ]:
# Run 2 -- reads seeded memory from the pipeline run before Section 7
agent = InvestmentResearchAgent(TICKER)
analysis_run2 = agent.run()

print()
print("=== Run 2 Results ===")
print(f"Evaluation score: {analysis_run2['evaluation_score']}/10")
print(f"Synthesis length: {len(analysis_run2['synthesis'])} characters")
print()
print(analysis_run2["synthesis"])

### Agent Function: Memory

Run 2 is complete. The analysis has been saved to `data/agent_memory.json`,
keyed by ticker. The memory entry includes the synthesis, evaluation score,
reflection, and timestamp. Run 3's planner will read this entry and incorporate
the prior reflection to address identified gaps.


In [ ]:
# Show memory saved by Run 2
prior = load_memory(TICKER)
if prior:
    print(f"Memory entry for {TICKER}:")
    print(f"  Timestamp:        {prior['timestamp']}")
    print(f"  Evaluation score: {prior['evaluation_score']}/10")
    print(f"  Synthesis length: {len(prior['synthesis'])} characters")
    print(f"  Reflection saved: {len(prior['reflection'])} characters")
else:
    print(f"No memory found for {TICKER}.")

### Run 3: Memory-Informed Analysis

The planner reads the memory saved by Run 2 and incorporates the prior
reflection into its research strategy, demonstrating Agent Function: Memory.


In [ ]:
# Run 3 -- planner reads Run 2 memory
analysis_run3 = agent.run()

print()
print("=== Run 3 Results ===")
print(f"Evaluation score: {analysis_run3['evaluation_score']}/10")
print(f"Synthesis length: {len(analysis_run3['synthesis'])} characters")
print()
print(analysis_run3["synthesis"])
print()
print("=== Score Comparison ===")
print(f"Run 1: {analysis_run1['evaluation_score']}/10")
print(f"Run 2: {analysis_run2['evaluation_score']}/10")
print(f"Run 3: {analysis_run3['evaluation_score']}/10")

### Observations

The Score Comparison above lists the evaluation score for each of the three
runs. A run that scores at or above the 7.0 threshold completes after a single
evaluation; a lower score triggers refinement, up to three iterations. Scores
come from an LLM evaluator, and re-scoring identical text can return a
different value (compare Run 1's score above with Step 1 of Section 7, which
evaluates the same saved synthesis), so differences between runs are not by
themselves evidence of improvement or degradation. The memory effect is shown
by planner behavior instead: the plan printed in Section 8 is generated from
Run 1's saved memory and addresses gaps identified in the prior analysis.


---

## 12. Test Suite

All contract tests pass without API calls, confirming the inter-agent schema,
routing logic, evaluator-optimizer loop, LLM retry behavior, and tool error
handling behave correctly in isolation.

In [ ]:
# Run the full test suite
import subprocess

result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-v", "--no-header", "-p", "no:cacheprovider"],
    capture_output=True,
    text=True,
)
# Strip the platform/path line from output
lines = result.stdout.splitlines()
filtered = [l for l in lines if not l.startswith("platform") and "/Users/" not in l]
print("\n".join(filtered))

---

## 13. References

Anthropic. (2026). Claude [Large language model]. https://claude.ai

Lee, N., Hong, J., & Thorne, J. (2024). Evaluating the consistency of LLM
evaluators. arXiv. https://arxiv.org/abs/2412.00543


<br><br>***AI Use Disclosure:*** *Claude (Anthropic, 2026), an AI language model, was used throughout this project to help design and write code and tests, debug failures, write code comments and markdown explanations, and review notebook outputs. The authors ran all code, reviewed and revised the AI's contributions, and are responsible for the final work.*